# B. Layout Aware Text Extraction (Computer Vision) — Pipeline End-to-End

**Tujuan (sesuai technical test):** mengubah gambar slide presentasi menjadi halaman HTML
di mana elemen teksnya dipisahkan dari elemen background — teks dirender sebagai `<span>`
yang diposisikan (top, left, width) berdasarkan *bounding box* hasil ekstraksi, lengkap dengan
style (warna & ukuran font), sehingga bisa diklik & diedit tanpa merusak background.

**Tahap pipeline:**
1. Downscale gambar (5734px → 2000px; posisi memakai persen → tetap akurat)
2. OCR (RapidOCR) → teks + bounding box + confidence
3. Deteksi style → warna teks, ukuran font, ketebalan
4. Inpainting (opsional — penanganan teks ganda) → background bersih
5. Generasi HTML (background layer + text layer overlay + Mode Edit)

> Jalankan dari folder `part_b_layout_extraction`:
> `pip install rapidocr-onnxruntime opencv-python-headless numpy ipython`

In [ ]:
import glob, sys
sys.path.insert(0, '.')
from IPython.display import display, Image as IPImage
from pipeline.extract import (load_and_ocr, extract_styles, inpaint_text,
                              generate_html, process_slide)
import cv2

SLIDES = sorted(glob.glob('../assets/AI Engineer Intern Technical Test/slides/*.jpg'))
print(f"{len(SLIDES)} slide ditemukan")

## Tahap 1–2 · Downscale + OCR (teks + bounding box)

In [ ]:
img, items, scale = load_and_ocr(SLIDES[0])   # slide pertama sebagai contoh
print(f"resolusi kerja: {img.shape[1]}x{img.shape[0]}  |  {len(items)} teks terdeteksi")
for it in items[:8]:
    print(f"  ({it.x:4d},{it.y:4d}) {it.w:4d}x{it.h:3d}px  conf={it.conf:.2f}  |  {it.text[:50]}")

## Tahap 3 · Deteksi Style (warna, ukuran font, ketebalan)

Warna = median piksel *tinta inti* (50% piksel terjauh dari warna background lokal) — tahan noise anti-aliasing. Ukuran font ≈ 0,78 × tinggi bounding box. Bold = densitas tinta.

In [ ]:
items = extract_styles(img, items)
print(f"{'teks':32s} {'warna':9s} {'font_px':>7s}  bold")
for it in items[:8]:
    print(f"{it.text[:32]:32s} {it.color:9s} {it.font_px:7.1f}  {'YA' if it.bold else '-'}")

## Tahap 4 · Inpainting — penanganan teks ganda (opsional/bonus)

Mask semua bounding box → `cv2.inpaint` → teks asli hilang dari background, sehingga overlay HTML tidak menduplikasi teks. (Alternatif CSS: `background-color` pada span — lihat `extract.py`.)

In [ ]:
cleaned = inpaint_text(img, items)
def show(arr, w=560):
    ok, buf = cv2.imencode('.jpg', arr); return IPImage(data=buf.tobytes(), width=w)
print('ASLI (teks menempel di gambar):'); display(show(img))
print('BERSIH (teks dihapus → siap di-overlay HTML):'); display(show(cleaned))

## Tahap 5 · Generasi HTML (background layer + text layer)

Setiap teks menjadi `<span>` absolut: `left/top/width` dalam persen, `font-size` dalam `cqw` (container query width) → proporsional di semua ukuran layar. File juga menyematkan background asli dengan tombol pengalih **🖼️ Background: Bersih/Asli** untuk membandingkan langsung, plus **✏️ Mode Edit** (contenteditable).

In [ ]:
out = generate_html(img, items, 'outputs/slide-01-demo.html', inpainted=cleaned)
print('ditulis:', out)

## Proses semua slide

In [ ]:
for i, f in enumerate(SLIDES, 1):
    info = process_slide(f, f'outputs/slide-{i:02d}.html')
    print(f"slide {i}: {info['texts']:2d} teks | conf rata2 {info['avg_conf']:.2f}")

## Kesesuaian dengan kriteria evaluasi

| Kriteria | Implementasi |
|---|---|
| Akurasi teks | RapidOCR + filter confidence (≥0,5) |
| Akurasi posisi | koordinat persen + `cqw` — QA headless Chrome: deviasi maks **0,96%** |
| Akurasi style | warna tinta inti + font-size + bold per teks |
| Penanganan teks ganda *(opsional)* | inpainting TELEA + tombol pembanding background |
| Edit teks *(opsional)* | Mode Edit (`contenteditable`) + Reset |